# FMP Earnings Call Transcript Extraction — MSCI World

Extracts earnings call transcripts for the MSCI World universe from Financial Modeling Prep (FMP), as a supplement to WRDS Capital IQ Transcripts (the thesis's primary planned source).

**How this notebook is organised.** All reusable logic lives in `fmp_utils.py` (same folder). The notebook only holds config + the calls that drive each step. When the logic needs changing, edit `fmp_utils.py` and re-run the **🔄 Reload** cell below — you do *not* need to re-run the whole notebook. Most steps also reconstruct their inputs from saved CSVs, so you can start from any step as long as the earlier steps have run at least once.

Pipeline: **Step 0** build ticker universe → **Step 1** coverage check → **Step 2** resolve ambiguous collisions → **Step 3** quantify coverage gap → **Step 4** bulk extraction.

In [3]:
import os
import importlib
from getpass import getpass
from pathlib import Path

import fmp_utils

# --- Paths (edit here only -- every step reads from these, never redefines
# its own copy, so any step can run standalone regardless of what ran before) ---
BASE_DIR = Path("/Users/zebakhoja/Documents/UCL Data Science & Public Policy MSc/UCL Thesis/AI Adoption")
CODE_DIR = BASE_DIR / "Code"
DATA_DIR = BASE_DIR / "Data"
OUT_DIR = DATA_DIR / "fmp_transcripts"          # per-transcript JSON files

ISHARES_SRC              = CODE_DIR / "MSCI World Index tickers.csv"
TICKER_LIST_PATH         = DATA_DIR / "msci_world_constituents.csv"
COLLISIONS_PATH          = DATA_DIR / "_ticker_collisions.csv"
COVERAGE_REPORT_PATH     = DATA_DIR / "_fmp_coverage_report.csv"
RESOLVED_COLLISIONS_PATH = DATA_DIR / "_ticker_collisions_resolved.csv"
UNCOVERED_RESOLVED_PATH  = DATA_DIR / "_uncovered_resolved.csv"
COUNTRY_TABLE_PATH       = DATA_DIR / "_country_coverage_table.csv"
EXTENDED_MAPPING_PATH    = DATA_DIR / "_extended_coverage_mapping.csv"
NEEDS_REVIEW_PATH        = DATA_DIR / "_needs_manual_review.csv"
MANUALLY_APPROVED_PATH   = DATA_DIR / "_manually_approved_mapping.csv"
MANIFEST_PATH            = OUT_DIR / "_manifest.csv"

REVIEW_FIELDNAMES = ["ticker", "original_ticker", "company_name", "country", "gics_sector",
                     "fmp_resolved_name", "fmp_resolved_exchange", "fmp_method",
                     "fmp_n_candidates", "resolution_similarity"]

OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- API key (prompted once; never written into the notebook) ---
FMP_API_KEY = os.environ.get("FMP_API_KEY") or getpass("Enter your FMP API key: ")

print("Config loaded. Now run the Reload cell below to create the client.")

Config loaded. Now run the Reload cell below to create the client.


### 🔄 Reload cell — re-run this after any edit to `fmp_utils.py`

Fast (no API calls, no key prompt). It re-imports the module and rebuilds the client, picking up code changes without re-running the whole notebook.

In [4]:
importlib.reload(fmp_utils)

client = fmp_utils.FMPClient(
    api_key=FMP_API_KEY,
    out_dir=OUT_DIR,
    request_delay=1.0,   # polite default; the bulk step lowers this
)
print("fmp_utils reloaded, client ready. api_call_count =", client.api_call_count)

fmp_utils reloaded, client ready. api_call_count = 0


## Step 0 — Build the ticker universe from iShares ETF holdings

`MSCI World Index tickers.csv`, exported from BlackRock's iShares Core MSCI World UCITS ETF holdings page, is a practical proxy for the MSCI World constituent list (WRDS doesn't carry MSCI index-membership data on this account). After filtering to equity holdings it is ~1,284 firms across 23 countries.

**Ticker collisions**: 44 rows share a bare ticker with a *different* company (e.g. `MRK` = Merck & Co US *and* Merck KGaA Germany). FMP resolves a bare symbol to its default (usually US) listing, so those non-US rows are flagged `fmp_ambiguous_symbol=True`, written to `_ticker_collisions.csv`, and excluded from the automatic pull — resolved separately in Step 2.

In [ ]:
summary = fmp_utils.build_ticker_list_from_ishares(
    ISHARES_SRC, TICKER_LIST_PATH, COLLISIONS_PATH
)
print(f"Equity holdings parsed : {summary['equity_holdings']}")
print(f"Ticker rows written    : {summary['ticker_rows']} ({summary['ambiguous']} flagged ambiguous)")
print(f"Collision detail       : {COLLISIONS_PATH.name} ({summary['collisions']} rows)")
print(f"Distinct countries     : {summary['countries']}")
print(f"Ticker list            : {TICKER_LIST_PATH}")

## Step 1 — Coverage check

Fetch FMP's master list of symbols that have *any* transcript, then split the universe into **covered** / **uncovered** / **ambiguous**. If the master list can't be fetched, everything non-ambiguous is treated as *uncovered* (never silently assumed covered) and a warning prints — re-run if you see it.

In [ ]:
msci_tickers = fmp_utils.load_ticker_list(TICKER_LIST_PATH)
print(f"Loaded {len(msci_tickers)} tickers.")

covered_symbols = client.fetch_covered_symbols()
master_list_available = bool(covered_symbols)

covered, uncovered, ambiguous = fmp_utils.split_universe(msci_tickers, covered_symbols)
fmp_utils.write_coverage_report(msci_tickers, covered_symbols, master_list_available, COVERAGE_REPORT_PATH)

if not master_list_available:
    print("\n  *** WARNING: master symbol list unavailable this run -- numbers below are "
          "unreliable; re-run this cell. ***")

flag = "" if master_list_available else "  [UNRELIABLE -- see warning]"
print(f"\nCovered  : {len(covered)}/{len(msci_tickers)} tickers have an FMP transcript on record.{flag}")
print(f"Uncovered: {len(uncovered)} (likely non-US firms with no ADR/dual-listing)")
print(f"Ambiguous (excluded, see _ticker_collisions.csv): {len(ambiguous)}")
print(f"Coverage report: {COVERAGE_REPORT_PATH}")

## Step 2 — Resolve the ambiguous (collision) tickers

For each flagged collision, look up what FMP actually resolves it to (`search-exchange-variants`, falling back to `search-name`), scoring candidates by name similarity + a US-exchange bonus. **Resumable** — writes each result immediately and skips anything already in the output CSV, so a crash/timeout mid-run doesn't lose progress. Review `_ticker_collisions_resolved.csv` before trusting any low-similarity match.

In [ ]:
# Standalone-safe: rebuild `ambiguous` from the collisions CSV if this cell is
# run without Step 1 in the current kernel.
try:
    ambiguous
except NameError:
    ambiguous = fmp_utils.load_ticker_list(COLLISIONS_PATH)
    print(f"(rebuilt `ambiguous` from {COLLISIONS_PATH.name}: {len(ambiguous)} rows)")

resolved_rows = client.run_symbol_resolution(ambiguous, RESOLVED_COLLISIONS_PATH)

high_confidence = [r for r in resolved_rows if r["resolved_symbol"] and r["similarity"] >= 0.5]
print(f"\nChecked {len(resolved_rows)} flagged rows.")
print(f"High-confidence resolution (similarity >= 0.5): {len(high_confidence)}")
print(f"Detail: {RESOLVED_COLLISIONS_PATH}")
for r in sorted(resolved_rows, key=lambda r: -r["similarity"])[:15]:
    print(f"  {r['ticker']:<6} '{r['expected_name']}' -> {r['resolved_symbol']} "
          f"'{r['resolved_name']}' ({r['resolved_exchange']}) sim={r['similarity']:.2f} via {r['method']}")

## Step 3 — Quantify the coverage gap (for the methods section)

Runs the same resolver over every *uncovered* ticker to separate a **true coverage gap** (FMP has no matching US-listed record — the disclosure-regime/language explanation) from a **likely ticker/symbol mismatch** (FMP has the company under a different symbol, recoverable). Breaks the true gap down by country. This is the quantified figure to cite rather than an estimate. Runs standalone from the saved coverage report if needed.

In [ ]:
SIMILARITY_THRESHOLD = 0.5

# Standalone-safe: rebuild `uncovered` from the saved coverage report.
try:
    uncovered
except NameError:
    uncovered = fmp_utils.load_coverage_subset(COVERAGE_REPORT_PATH, "False")
    print(f"(rebuilt `uncovered` from {COVERAGE_REPORT_PATH.name}: {len(uncovered)} rows)")

uncovered_resolved = client.run_symbol_resolution(uncovered, UNCOVERED_RESOLVED_PATH)
likely_mismatch, true_gap, gap_countries = fmp_utils.classify_uncovered(
    uncovered_resolved, SIMILARITY_THRESHOLD
)

n = len(uncovered_resolved)
print(f"\nUncovered tickers checked: {n}")
if n:
    print(f"  Likely ticker/symbol mismatch (US match, sim >= {SIMILARITY_THRESHOLD}): "
          f"{len(likely_mismatch)} ({len(likely_mismatch)/n:.1%})")
    print(f"  True coverage gap (no US match found): {len(true_gap)} ({len(true_gap)/n:.1%})")
print(f"Detail: {UNCOVERED_RESOLVED_PATH}")

print("\nTrue-gap countries (most affected):")
for country, cnt in gap_countries.most_common(15):
    print(f"  {country:<20} {cnt}")

### Step 3b — Full per-country coverage table

Breaks all 23 countries down into covered / likely-mismatch / true-gap / ambiguous counts and rates, ready to drop into a thesis table. Standalone-safe: rebuilds `covered`, `ambiguous`, `likely_mismatch`, `true_gap` from saved CSVs if they're not already in memory.

In [ ]:
try:
    msci_tickers
except NameError:
    msci_tickers = fmp_utils.load_ticker_list(TICKER_LIST_PATH)
try:
    covered
except NameError:
    covered = fmp_utils.load_coverage_subset(COVERAGE_REPORT_PATH, "True")
try:
    ambiguous
except NameError:
    ambiguous = fmp_utils.load_ticker_list(COLLISIONS_PATH)
try:
    likely_mismatch, true_gap
except NameError:
    uncovered_resolved = fmp_utils.load_resolved(UNCOVERED_RESOLVED_PATH)
    likely_mismatch, true_gap, _ = fmp_utils.classify_uncovered(
        uncovered_resolved, globals().get("SIMILARITY_THRESHOLD", 0.5)
    )

country_table = fmp_utils.build_country_coverage_table(msci_tickers, covered, ambiguous, likely_mismatch, true_gap)
fmp_utils.write_rows_csv(country_table, COUNTRY_TABLE_PATH)

print(f"{'Country':<18} {'Total':>6} {'Covered':>9} {'Mismatch':>9} {'TrueGap':>9} {'Ambig':>7}")
for row in country_table:
    print(f"{row['country']:<18} {row['total']:>6} "
          f"{row['covered']:>5} ({row['covered_rate']:>5.0%}) "
          f"{row['likely_mismatch']:>5} ({row['mismatch_rate']:>5.0%}) "
          f"{row['true_gap']:>5} ({row['true_gap_rate']:>5.0%}) "
          f"{row['ambiguous']:>4} ({row['ambiguous_rate']:>4.0%})")
print(f"\nSaved: {COUNTRY_TABLE_PATH}")

## Step 4 — Bulk transcript extraction

Fetches transcripts for every covered ticker across 2018–present. **Resumable** (skips already-saved JSON), **budget-aware** (stops cleanly at `max_calls_per_run`), logs every attempt to `_manifest.csv`, and splits prepared remarks vs Q&A with the heuristic in `fmp_utils`.

If it stops on the budget: call `client.reset_budget(...)` (already at the top of the cell) and re-run — done work is skipped. Tune `max_calls_per_run` to your FMP plan's daily limit.

In [ ]:
client.reset_budget(max_calls_per_run=4000, request_delay=0.3)

# Standalone-safe: rebuild `covered` from the saved coverage report.
try:
    covered
except NameError:
    covered = fmp_utils.load_coverage_subset(COVERAGE_REPORT_PATH, "True")
    print(f"(rebuilt `covered` from {COVERAGE_REPORT_PATH.name}: {len(covered)} rows)")

result = client.run_bulk_extraction(covered, MANIFEST_PATH)

budget_str = str(result["budget"]) if result["budget"] else "unlimited"
print(f"\nRun ended. API calls this run: {result['api_calls']} (budget: {budget_str})")
print(f"Manifest log: {MANIFEST_PATH}")
if result["budget_hit"]:
    print("Stopped on budget. Re-run this cell (reset_budget runs automatically) to continue.")

## Step 5 — Extend coverage using resolved symbols

Of the 244 "likely mismatch" tickers from Step 3, this fetches transcripts for the ones FMP resolved with **high confidence** (similarity ≥ `AUTO_THRESHOLD`) under their *resolved* symbol instead of the original MSCI ticker — recovering genuine coverage rather than leaving them as gaps. Moderate-confidence rows (`REVIEW_THRESHOLD` ≤ similarity < `AUTO_THRESHOLD`) are **not** auto-fetched — they're written to `_needs_manual_review.csv` for you to check by hand before deciding whether to include them (same reasoning as the ticker-collision handling: don't silently attach a transcript to the wrong company).

`_extended_coverage_mapping.csv` records `original_ticker -> resolved_symbol` for every row actually fetched, so you can trace back to `gics_sector`/`country` from the original MSCI ticker list when merging with accounting data later. Saved transcript files are still named by the *resolved* symbol (e.g. `SNY_2023_Q2.json`), consistent with Step 4.

In [ ]:
AUTO_THRESHOLD = 0.85
REVIEW_THRESHOLD = 0.5

try:
    msci_tickers
except NameError:
    msci_tickers = fmp_utils.load_ticker_list(TICKER_LIST_PATH)
try:
    covered
except NameError:
    covered = fmp_utils.load_coverage_subset(COVERAGE_REPORT_PATH, "True")
try:
    uncovered_resolved
except NameError:
    uncovered_resolved = fmp_utils.load_resolved(UNCOVERED_RESOLVED_PATH)

auto_include, needs_review = fmp_utils.build_extended_coverage(
    uncovered_resolved, msci_tickers, AUTO_THRESHOLD, REVIEW_THRESHOLD
)
auto_include = fmp_utils.dedupe_by_ticker(covered, auto_include)  # skip if already covered

# fmp_resolved_name/fmp_resolved_exchange are FMP's own record of what the
# resolved symbol actually is -- side by side with the iShares company_name,
# this is what you need to judge the match, not just the similarity score.
fmp_utils.write_rows_csv(auto_include, EXTENDED_MAPPING_PATH, fieldnames=REVIEW_FIELDNAMES)
fmp_utils.write_rows_csv(needs_review, NEEDS_REVIEW_PATH, fieldnames=REVIEW_FIELDNAMES)

print(f"Auto-include (similarity >= {AUTO_THRESHOLD}): {len(auto_include)} -- fetching now")
print(f"Needs manual review ({REVIEW_THRESHOLD} <= similarity < {AUTO_THRESHOLD}): "
      f"{len(needs_review)} -- saved to {NEEDS_REVIEW_PATH.name}, not fetched")

client.reset_budget(max_calls_per_run=4000, request_delay=0.3)
result = client.run_bulk_extraction(auto_include, MANIFEST_PATH)

budget_str = str(result["budget"]) if result["budget"] else "unlimited"
print(f"\nRun ended. API calls this run: {result['api_calls']} (budget: {budget_str})")
print(f"Mapping saved: {EXTENDED_MAPPING_PATH}")
if result["budget_hit"]:
    print("Stopped on budget. Re-run this cell to continue (already-saved transcripts are skipped).")

### Step 5b — Fetch the manually-approved review list

After eyeballing `_needs_manual_review.csv` (iShares name/country vs. FMP's own `fmp_resolved_name`/`fmp_resolved_exchange` side by side), all 90 rows were confirmed as legitimate matches — mostly the `CORP`→`Corporation` / home-market-ticker→OTC-ADR pattern seen in the Japanese names (Inpex, Taisei, Obayashi, Shimizu, Kajima, Kikkoman, Asahi Kasei, etc.), not genuine mismatches. Fetches transcripts for the full list under their resolved symbols and records them in `_manually_approved_mapping.csv`, kept separate from `_extended_coverage_mapping.csv` (the auto-included ≥0.85 tier) so the two provenance tiers stay distinguishable later — e.g. for reporting "244 auto-matched vs. 90 manually verified" rather than blurring them into one number.

In [5]:
try:
    needs_review
except NameError:
    needs_review = fmp_utils.load_ticker_list(NEEDS_REVIEW_PATH)
    print(f"(rebuilt `needs_review` from {NEEDS_REVIEW_PATH.name}: {len(needs_review)} rows)")

fmp_utils.write_rows_csv(needs_review, MANUALLY_APPROVED_PATH, fieldnames=REVIEW_FIELDNAMES)
print(f"Manually approved: {len(needs_review)} tickers -- fetching now")

client.reset_budget(max_calls_per_run=4000, request_delay=0.3)
result = client.run_bulk_extraction(needs_review, MANIFEST_PATH)

budget_str = str(result["budget"]) if result["budget"] else "unlimited"
print(f"\nRun ended. API calls this run: {result['api_calls']} (budget: {budget_str})")
print(f"Mapping saved: {MANUALLY_APPROVED_PATH}")
if result["budget_hit"]:
    print("Stopped on budget. Re-run this cell to continue (already-saved transcripts are skipped).")

(rebuilt `needs_review` from _needs_manual_review.csv: 90 rows)
Manually approved: 90 tickers -- fetching now

Run ended. API calls this run: 1148 (budget: 4000)
Mapping saved: /Users/zebakhoja/Documents/UCL Data Science & Public Policy MSc/UCL Thesis/AI Adoption/Data/_manually_approved_mapping.csv
